# Edmonton Bike-Share Data Exploration

This notebook explores the key datasets we'll use to plan a bike-sharing network for Edmonton, AB:

1. **Street network** from OpenStreetMap via OSMnx
2. **Bike infrastructure** from Edmonton Open Data
3. **Neighbourhood boundaries** and population
4. **H3 hex grid** for demand aggregation
5. **Points of interest** (trip generators)

Make sure you've installed the backend dependencies first:
```bash
cd backend && pip install -e .
```

In [ ]:
import sys
sys.path.insert(0, '../backend')

import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import pydeck as pdk
import pandas as pd
import numpy as np

from src.config import EDMONTON_PLACE, EDMONTON_CENTER

print(f"OSMnx version: {ox.__version__}")
print(f"Target: {EDMONTON_PLACE}")
print(f"Center: {EDMONTON_CENTER}")

## 1. Edmonton Street Network (OSMnx)

Download the cycling-usable street network from OpenStreetMap.

In [ ]:
# Fetch the bike-able network graph
G_bike = ox.graph_from_place(EDMONTON_PLACE, network_type='bike')
print(f"Nodes: {G_bike.number_of_nodes():,}")
print(f"Edges: {G_bike.number_of_edges():,}")

# Quick matplotlib plot
fig, ax = ox.plot_graph(G_bike, figsize=(12, 12), node_size=0, edge_linewidth=0.3, bgcolor='#0a0a0a')
plt.title('Edmonton — Cycling Network (OSM)', color='white', fontsize=14)
plt.show()

## 2. City Boundary & Neighbourhoods

In [ ]:
from src.data.edmonton import get_edmonton_boundary

boundary = get_edmonton_boundary()
boundary.plot(figsize=(8, 8), edgecolor='cyan', facecolor='none', linewidth=2)
plt.title('Edmonton City Boundary')
plt.axis('off')
plt.show()

print(f"Area: {boundary.to_crs('EPSG:32612').area.values[0] / 1e6:.1f} km²")

## 3. H3 Hex Grid

Generate a hexagonal grid over Edmonton for demand aggregation and station candidate generation.

In [ ]:
from src.data.hexgrid import generate_hex_grid

# Resolution 8 gives ~0.74 km² hexagons — good for city-scale planning
hex_grid = generate_hex_grid(boundary, resolution=8)
print(f"Generated {len(hex_grid)} hex cells at resolution 8")

hex_grid.head()

In [ ]:
# Visualize the hex grid with PyDeck
view = pdk.ViewState(
    latitude=EDMONTON_CENTER[0],
    longitude=EDMONTON_CENTER[1],
    zoom=10,
    pitch=45,
)

hex_layer = pdk.Layer(
    'H3HexagonLayer',
    data=hex_grid[['h3_index']].rename(columns={'h3_index': 'hex'}),
    get_hexagon='hex',
    get_fill_color=[0, 180, 200, 80],
    get_line_color=[255, 255, 255, 40],
    line_width_min_pixels=1,
    extruded=False,
    pickable=True,
)

pdk.Deck(layers=[hex_layer], initial_view_state=view, map_style='dark')

## 4. Points of Interest (Trip Generators)

Pull POIs from OSM that are likely origins/destinations for bike trips.

In [ ]:
from src.data.edmonton import fetch_pois

pois = fetch_pois(use_cache=True)
print(f"Fetched {len(pois)} POIs")

# Show the top amenity types
if 'amenity' in pois.columns:
    print("\nTop amenity types:")
    print(pois['amenity'].value_counts().head(15))

## 5. Edmonton Open Data — Traffic & Bike Infra

In [ ]:
from src.data.edmonton import fetch_traffic_volumes, fetch_bike_infrastructure

traffic = fetch_traffic_volumes(use_cache=True)
print(f"Traffic volume records: {len(traffic)}")
print(traffic.columns.tolist())
traffic.head()

In [ ]:
bike_infra = fetch_bike_infrastructure(use_cache=True)
print(f"Bike infrastructure records: {len(bike_infra)}")
print(bike_infra.columns.tolist())
bike_infra.head()

## 6. Quick Simulation Test

Run a small simulation with a few manually placed stations to verify the engine works.

In [ ]:
from src.simulation.engine import BikeShareSimulation, Station

# Place 5 test stations in central Edmonton
test_stations = [
    Station('s1', 'Downtown/Jasper Ave', 53.5409, -113.4938, capacity=30, initial_bikes=15),
    Station('s2', 'University of Alberta', 53.5232, -113.5263, capacity=25, initial_bikes=12),
    Station('s3', 'Whyte Avenue', 53.5178, -113.4977, capacity=20, initial_bikes=10),
    Station('s4', 'NAIT', 53.5681, -113.5054, capacity=15, initial_bikes=8),
    Station('s5', 'Ice District', 53.5470, -113.4970, capacity=25, initial_bikes=12),
]

# Simple OD matrix: uniform demand between all pairs
od = {}
for s1 in test_stations:
    for s2 in test_stations:
        if s1.id != s2.id:
            od[(s1.id, s2.id)] = 1.0

sim = BikeShareSimulation(
    stations=test_stations,
    od_matrix=od,
    duration_hours=24,
    trips_per_hour=50,
    avg_trip_minutes=15,
)

results = sim.run()
print(f"Total trips attempted: {results['total_trips']}")
print(f"Successful: {results['successful_trips']}")
print(f"Failed: {results['failed_trips']}")
print(f"Service rate: {results['service_rate']:.1%}")
print(f"Avg trip duration: {results['avg_trip_duration']:.1f} min")

print("\nStation Summary:")
for sid, stats in results['station_stats'].items():
    print(f"  {stats['name']:25s}  bikes={stats['final_bikes']:2d}/{stats['capacity']:2d}  "
          f"started={stats['trips_started']:3d}  ended={stats['trips_ended']:3d}  "
          f"net_flow={stats['net_flow']:+d}")

## Next Steps

- Weight OD matrix by POI density, population, and transit connectivity
- Use OR-Tools optimization to find optimal station locations
- Apply time-varying demand patterns (morning/evening commute peaks)
- Model rebalancing trucks
- Visualize simulation results in the frontend